# OD Reading vs. Hours Since Experiment

This notebook lets you load a CSV file and plot `od_reading` vs `hours_since_experiment_created`, with one trace per pioreactor unit.

It also includes optional normalization cells so you can compare trajectories across units after baseline scaling. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

from scipy.signal import medfilt
from scipy.stats import linregress
import numpy as np 

## 1. Select File 

In [ ]:
import os, sys

# --- Auto-detect environment ---
IN_COLAB = 'google.colab' in sys.modules or os.environ.get('COLAB_RELEASE_TAG') is not None

if IN_COLAB:
    from google.colab import files
    from pathlib import Path
    print("Running in Google Colab — please upload your CSV file:")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    CSVPATH = Path(fname)
else:
    import tkinter as tk
    from tkinter import filedialog
    from pathlib import Path
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    CSVPATH = filedialog.askopenfilename(
        title='Select CSV file',
        filetypes=[('CSV files', '*.csv'), ('All files', '*.*')]
    )
    root.destroy()
    if not CSVPATH:
        raise FileNotFoundError('No file was selected.')
    CSVPATH = Path(CSVPATH).expanduser().resolve()

print(f'Selected file: {CSVPATH}')

## 2. Load Data 

In [ ]:
df = pd.read_csv(CSVPATH, parse_dates=['timestamp', 'timestamp_localtime'])

# Rename for convenience
df = df.rename(columns={
    'hours_since_experiment_created': 'hourssince',
    'pioreactor_unit': 'pioreactor',
})

print(f'Rows: {len(df):,}  Columns: {list(df.columns)}')
df.head(3)

# Save all output figures next to the source CSV
OUTPUTDIR = CSVPATH.parent
print(f'Output directory: {OUTPUTDIR}')

## 3. Optional Downsample for Fast Plotting

The dataset can be large. If plotting is slow, uncomment the downsample cell below
to keep every Nth point per unit (a rolling median is applied first to reduce noise). 

In [ ]:
# --- Uncomment to downsample ---
# DOWNSAMPLE_N = 10  # keep 1 in every N points per pioreactor
# df = (
#     df.sort_values(['pioreactor', 'hourssince'])
#     .groupby('pioreactor', group_keys=False)
#     .apply(lambda g: g.iloc[::DOWNSAMPLE_N])
#     .reset_index(drop=True)
# )
# print(f'After downsampling: {len(df)} rows')

## 4. Filter by Channel (if needed) 

In [ ]:
# Set to None to keep all channels, or e.g. CHANNEL = 2
CHANNEL = None

if CHANNEL is not None:
    dfplot = df[df['channel'] == CHANNEL].copy()
    print(f'Filtered to channel {CHANNEL}: {len(dfplot)} rows')
else:
    dfplot = df.copy()
    print(f'Using all channels: {len(dfplot)} rows')

## 5. Create Normalized OD Values

This section creates normalized versions of `od_reading` for each pioreactor.

Included normalizations:
- `od_norm_t0`: divides by the first OD value for each pioreactor.
- `od_delta_t0`: subtracts the first OD value for each pioreactor.
- `od_minmax`: rescales each pioreactor trace between 0 and 1. 

In [ ]:
dfplot = dfplot.sort_values(['pioreactor', 'hourssince']).copy()

first_od  = dfplot.groupby('pioreactor')['od_reading'].transform('first')
min_od    = dfplot.groupby('pioreactor')['od_reading'].transform('min')
max_od    = dfplot.groupby('pioreactor')['od_reading'].transform('max')
range_od  = (max_od - min_od).replace(0, pd.NA)

dfplot['od_norm_t0']  = dfplot['od_reading'] / first_od
dfplot['od_delta_t0'] = dfplot['od_reading'] - first_od
dfplot['od_minmax']   = (dfplot['od_reading'] - min_od) / range_od

cols_to_show = ['pioreactor', 'hourssince', 'od_reading', 'od_norm_t0', 'od_delta_t0', 'od_minmax']
dfplot[cols_to_show].head()

## 6. Plot Normalized OD Relative to First Time Point 

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for unit, grp in dfplot.groupby('pioreactor'):
    ax.plot(grp['hourssince'], grp['od_norm_t0'], label=unit, alpha=0.8)
ax.set_xlabel('Hours since experiment created')
ax.set_ylabel('OD / OD(t=0)')
ax.set_title('Normalized OD (relative to first reading)')
ax.legend(title='Unit', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_norm_t0.png', dpi=150)
plt.show()

## 7. Plot Raw OD — All Units Overlay 

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for unit, grp in dfplot.groupby('pioreactor'):
    ax.plot(grp['hourssince'], grp['od_reading'], label=unit, alpha=0.7, linewidth=0.8)
ax.set_xlabel('Hours since experiment created')
ax.set_ylabel('od_reading')
ax.set_title('od_reading vs. Time — All Pioreactor Units')
ax.legend(title='Unit', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_vs_hours_all_units.png', dpi=150)
plt.show()

## 8. Plot Raw OD — Individual Unit Subplots 

In [ ]:
units = sorted(dfplot['pioreactor'].unique())
ncols = 3
nrows = -(-len(units) // ncols)  # ceiling division
palette = sns.color_palette('tab10', len(units))

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), sharey=False)
axes = axes.flatten()

for i, (unit, color) in enumerate(zip(units, palette)):
    grp = dfplot[dfplot['pioreactor'] == unit]
    axes[i].plot(grp['hourssince'], grp['od_reading'], color=color, linewidth=0.8)
    axes[i].set_title(unit, fontweight='bold')
    axes[i].set_xlabel('Hours since start')
    axes[i].set_ylabel('od_reading')
    axes[i].grid(True, linestyle='--', alpha=0.5)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle('od_reading vs. Time — Individual Units', fontweight='bold', fontsize=14)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_vs_hours_subplots.png', dpi=150)
plt.show()

## 9. Smoothed OD — Rolling Median Overlay 

In [ ]:
SMOOTH_WIN = 60

fig, ax = plt.subplots(figsize=(14, 5))
palette = sns.color_palette('tab10', len(units))

for unit, color in zip(units, palette):
    grp = dfplot[dfplot['pioreactor'] == unit].sort_values('hourssince')
    smoothed = grp['od_reading'].rolling(SMOOTH_WIN, center=True, min_periods=1).median()
    ax.plot(grp['hourssince'], grp['od_reading'], color=color, alpha=0.2, linewidth=0.5)
    ax.plot(grp['hourssince'], smoothed, color=color, linewidth=1.5, label=unit)

ax.set_xlabel('Hours since experiment created')
ax.set_ylabel('od_reading')
ax.set_title(f'od_reading vs. Time — Smoothed (rolling median, window={SMOOTH_WIN})')
ax.legend(title='Unit', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_vs_hours_smoothed.png', dpi=150)
plt.show()

## 10. OD Doubling Time — Exponential Fit

Finds the best local exponential growth window per unit using a sliding window on log(OD),
then reports growth rate *r* (h⁻¹), doubling time *Td* = ln(2)/r, and R². 

In [ ]:
WIN_H    = 3.0   # sliding window width in hours
MIN_OD   = 0.05  # ignore readings below this (noise floor)
MEDFILT_K = 51   # robust smoothing kernel (must be odd)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), sharey=False)
axes = axes.flatten()
palette = sns.color_palette('tab10', len(units))

for i, (unit, color) in enumerate(zip(units, palette)):
    grp = dfplot[dfplot['pioreactor'] == unit].sort_values('hourssince').copy()
    grp = grp[grp['od_reading'] > MIN_OD].reset_index(drop=True)
    if len(grp) < 20:
        continue

    t  = grp['hourssince'].values
    od = grp['od_reading'].values

    # Robust median filter
    k  = min(MEDFILT_K, len(od) if len(od) % 2 == 1 else len(od) - 1)
    od_s = medfilt(od, kernel_size=k)

    # Sliding window linear regression on log(OD)
    best_r2, best_r, best_td, best_mask = -np.inf, np.nan, np.nan, None
    log_od = np.log(od_s)
    for j in range(len(t)):
        mask = (t >= t[j]) & (t < t[j] + WIN_H)
        if mask.sum() < 5:
            continue
        slope, intercept, r, p, se = linregress(t[mask], log_od[mask])
        if r**2 > best_r2 and slope > 0:
            best_r2 = r**2
            best_r  = slope
            best_td = np.log(2) / slope
            best_mask = mask
            best_intercept = intercept

    ax = axes[i]
    ax.plot(t, od, color=color, alpha=0.2, linewidth=0.5)
    ax.plot(t, od_s, color=color, linewidth=1.5)

    if best_mask is not None:
        t_win = t[best_mask]
        ax.fill_betweenx([0, od.max() * 1.1], t_win[0], t_win[-1],
                         color='gray', alpha=0.15)
        ax.plot(t_win, np.exp(best_intercept + best_r * t_win),
                'k--', linewidth=1.5, label='Exp fit')
        ax.text(0.02, 0.97,
                f'r={best_r:.3f} h\u207b\u00b9\nTd={best_td:.2f} h\nR\u00b2={best_r2:.3f}',
                transform=ax.transAxes, va='top', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

    ax.set_title(unit, fontweight='bold')
    ax.set_xlabel('Hours since experiment created')
    ax.set_ylabel('od_reading')
    ax.grid(True, linestyle='--', alpha=0.5)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle('od_reading with robust smoothing and best local exponential fit',
             fontweight='bold', fontsize=13)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_robust_expfit_subplots.png', dpi=150)
plt.show()